# SHAP 분석: 부도예측 모델 해석

## 목적
학습된 XGBoost 모델의 예측을 SHAP(SHapley Additive exPlanations)으로 해석합니다.

## 분석 내용
1. **SHAP Summary Plot (Beeswarm)**: 전체 피처 중요도와 방향성
2. **SHAP Bar Plot**: 평균 절대 SHAP값 기반 중요도
3. **SHAP Waterfall Plot**: 개별 예측 설명
4. **SHAP Dependence Plot**: 주요 피처별 영향도
5. **SHAP Force Plot**: 전체 데이터셋 시각화

In [ ]:
# 환경 설정
import sys
from pathlib import Path

# 프로젝트 루트 추가
project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 8)

# 출력 디렉토리
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print("SHAP version:", shap.__version__)
print("Output directory:", OUTPUT_DIR.resolve())

## 1. 모델 및 데이터 로드

In [ ]:
# 학습된 모델 로드
model_dir = Path('../../models/default_prediction')

with open(model_dir / 'model.pkl', 'rb') as f:
    model = pickle.load(f)

with open(model_dir / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

import json
with open(model_dir / 'metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"Model type: {metadata['model_type']}")
print(f"Number of features: {metadata['n_features']}")
print(f"AUC-ROC: {metadata['metrics']['auc_roc']:.4f}")

In [ ]:
# Feature Store에서 테스트 데이터 로드
from ml.common.feature_store import load_from_feature_store

# 테스트 기간 데이터
test_df = load_from_feature_store(base_ym=[20220401], include_target=True)

# 컬럼명 대문자로 변환 (scaler가 대문자로 학습됨)
test_df.columns = [c.upper() if c not in ['base_ym', 'company_id', 'sic_cd_3', 'default_yn'] else c for c in test_df.columns]

# 피처 컬럼 (메타데이터에서)
feature_cols = metadata['feature_names']

# 피처와 타겟 분리
X_test = test_df[feature_cols].copy()
y_test = test_df['default_yn'].copy()

# 결측치 처리 (중앙값)
for col in feature_cols:
    if X_test[col].isnull().any():
        X_test[col] = X_test[col].fillna(X_test[col].median())

# 스케일링
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"테스트 데이터: {len(X_test):,}개 기업")
print(f"부도 기업: {y_test.sum():,}개 ({y_test.mean()*100:.2f}%)")

## 2. SHAP 값 계산

In [ ]:
# SHAP Explainer 생성 (TreeExplainer for XGBoost)
explainer = shap.TreeExplainer(model)

# SHAP 값 계산 (샘플링하여 속도 개선)
sample_size = min(2000, len(X_test_scaled_df))
np.random.seed(42)
sample_idx = np.random.choice(len(X_test_scaled_df), sample_size, replace=False)

X_sample = X_test_scaled_df.iloc[sample_idx]
y_sample = y_test.iloc[sample_idx]

print(f"SHAP 계산 샘플 크기: {sample_size:,}개")
print("SHAP 값 계산 중...")

shap_values = explainer.shap_values(X_sample)

print(f"SHAP values shape: {shap_values.shape}")
print("SHAP 값 계산 완료!")

## 3. SHAP Summary Plot (Beeswarm)

모든 피처의 SHAP 값 분포를 보여줍니다.
- **X축**: SHAP 값 (예측에 대한 기여도)
- **Y축**: 피처 (중요도 순 정렬)
- **색상**: 피처 값의 크기 (빨강=높음, 파랑=낮음)

In [ ]:
# SHAP Summary Plot (Beeswarm)
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
plt.title('SHAP Summary Plot: Feature Importance & Direction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_summary_beeswarm.png'}")

## 4. SHAP Bar Plot (Mean Absolute SHAP)

각 피처의 평균 절대 SHAP 값으로 전체 중요도를 보여줍니다.

In [ ]:
# SHAP Bar Plot
plt.figure(figsize=(10, 10))
shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False, max_display=20)
plt.title('SHAP Feature Importance (Mean |SHAP|)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_bar_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_bar_importance.png'}")

In [ ]:
# 피처 중요도 테이블
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("\nSHAP 기반 피처 중요도 Top 20:")
print("="*50)
for i, row in importance_df.head(20).iterrows():
    print(f"{row['feature']:15} : {row['mean_abs_shap']:.4f}")

# CSV 저장
importance_df.to_csv(OUTPUT_DIR / 'shap_feature_importance.csv', index=False)
print(f"\n저장: {OUTPUT_DIR / 'shap_feature_importance.csv'}")

## 5. SHAP Waterfall Plot (Individual Prediction)

개별 예측이 어떻게 만들어졌는지 설명합니다.

In [ ]:
# 부도 기업 중 하나 선택
default_idx = y_sample[y_sample == 1].index
if len(default_idx) > 0:
    sample_default_idx = default_idx[0]
    sample_pos = y_sample.index.get_loc(sample_default_idx)
    
    print(f"선택된 부도 기업 인덱스: {sample_default_idx}")
    print(f"예측 확률: {model.predict_proba(X_sample.iloc[[sample_pos]])[0][1]:.4f}")
    print(f"실제 값: 부도")
else:
    sample_pos = 0
    print("부도 기업이 샘플에 없어 첫 번째 기업 사용")

In [ ]:
# Waterfall Plot - 부도 기업
plt.figure(figsize=(10, 8))

# shap.Explanation 객체 생성
explanation = shap.Explanation(
    values=shap_values[sample_pos],
    base_values=explainer.expected_value,
    data=X_sample.iloc[sample_pos].values,
    feature_names=feature_cols
)

shap.waterfall_plot(explanation, max_display=15, show=False)
plt.title('SHAP Waterfall: Default Company Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_waterfall_default.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_waterfall_default.png'}")

In [ ]:
# Waterfall Plot - 정상 기업
normal_idx = y_sample[y_sample == 0].index
if len(normal_idx) > 0:
    sample_normal_idx = normal_idx[0]
    normal_pos = y_sample.index.get_loc(sample_normal_idx)
    
    print(f"선택된 정상 기업 인덱스: {sample_normal_idx}")
    print(f"예측 확률: {model.predict_proba(X_sample.iloc[[normal_pos]])[0][1]:.4f}")
    print(f"실제 값: 정상")

    plt.figure(figsize=(10, 8))
    explanation_normal = shap.Explanation(
        values=shap_values[normal_pos],
        base_values=explainer.expected_value,
        data=X_sample.iloc[normal_pos].values,
        feature_names=feature_cols
    )
    shap.waterfall_plot(explanation_normal, max_display=15, show=False)
    plt.title('SHAP Waterfall: Normal Company Prediction', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'shap_waterfall_normal.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n저장: {OUTPUT_DIR / 'shap_waterfall_normal.png'}")

## 6. SHAP Dependence Plots

주요 피처가 예측에 미치는 영향을 상세히 분석합니다.

In [ ]:
# 상위 6개 피처에 대한 Dependence Plot
top_features = importance_df.head(6)['feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    feature_idx = feature_cols.index(feature)
    
    shap.dependence_plot(
        feature_idx, 
        shap_values, 
        X_sample, 
        ax=ax, 
        show=False,
        interaction_index='auto'
    )
    ax.set_title(f'{feature}', fontsize=12, fontweight='bold')

plt.suptitle('SHAP Dependence Plots: Top 6 Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_dependence_top6.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_dependence_top6.png'}")

## 7. SHAP 값 분포 분석

In [ ]:
# 부도/정상 기업별 SHAP 값 분포 비교
top_5_features = importance_df.head(5)['feature'].tolist()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, feature in enumerate(top_5_features):
    ax = axes[idx]
    feature_idx = feature_cols.index(feature)
    
    # 부도/정상별 SHAP 값
    shap_default = shap_values[y_sample.values == 1, feature_idx]
    shap_normal = shap_values[y_sample.values == 0, feature_idx]
    
    ax.hist(shap_normal, bins=30, alpha=0.6, label='Normal', color='green', density=True)
    ax.hist(shap_default, bins=30, alpha=0.6, label='Default', color='red', density=True)
    ax.axvline(0, color='black', linestyle='--', alpha=0.5)
    ax.set_title(f'{feature}', fontsize=11, fontweight='bold')
    ax.set_xlabel('SHAP Value')
    if idx == 0:
        ax.set_ylabel('Density')
        ax.legend()

plt.suptitle('SHAP Value Distribution: Default vs Normal', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_distribution_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_distribution_by_class.png'}")

## 8. SHAP Force Plot (전체 데이터셋)

In [ ]:
# Force plot (단일 샘플) - matplotlib 지원
# 다중 샘플 force plot은 matplotlib=True를 지원하지 않음

# 부도 기업 force plot
plt.figure(figsize=(14, 3))
shap.force_plot(
    explainer.expected_value, 
    shap_values[sample_pos], 
    X_sample.iloc[sample_pos],
    show=False,
    matplotlib=True
)
plt.title('SHAP Force Plot: Default Company', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_force_plot_default.png', dpi=150, bbox_inches='tight')
plt.show()

# 정상 기업 force plot
plt.figure(figsize=(14, 3))
shap.force_plot(
    explainer.expected_value, 
    shap_values[normal_pos], 
    X_sample.iloc[normal_pos],
    show=False,
    matplotlib=True
)
plt.title('SHAP Force Plot: Normal Company', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_force_plot_normal.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_force_plot_default.png'}")
print(f"저장: {OUTPUT_DIR / 'shap_force_plot_normal.png'}")

## 9. 피처 그룹별 SHAP 중요도

In [ ]:
# 피처 그룹 정의
feature_groups = {
    'FN1 (자산/자본)': [f for f in feature_cols if f.startswith('FN1')],
    'FN2 (부채)': [f for f in feature_cols if f.startswith('FN2')],
    'FN3 (손익)': [f for f in feature_cols if f.startswith('FN3')],
    'DA/DB (재무상태)': [f for f in feature_cols if f.startswith('DA') or f.startswith('DB') or f.startswith('D2')],
    'R (비율지표)': [f for f in feature_cols if f.startswith('R')],
    'N (파생비율)': [f for f in feature_cols if f.startswith('N')]
}

# 그룹별 SHAP 합계
group_importance = {}
for group_name, group_features in feature_groups.items():
    if group_features:
        group_idx = [feature_cols.index(f) for f in group_features if f in feature_cols]
        if group_idx:
            group_shap = np.abs(shap_values[:, group_idx]).sum(axis=1).mean()
            group_importance[group_name] = group_shap

# 시각화
plt.figure(figsize=(10, 6))
groups = list(group_importance.keys())
values = list(group_importance.values())
colors = plt.cm.Set2(np.linspace(0, 1, len(groups)))

bars = plt.barh(groups, values, color=colors)
plt.xlabel('Mean |SHAP| Value')
plt.title('Feature Group Importance (SHAP)', fontsize=14, fontweight='bold')

# 값 표시
for bar, val in zip(bars, values):
    plt.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_group_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n저장: {OUTPUT_DIR / 'shap_group_importance.png'}")

## 10. 분석 요약

In [ ]:
print("="*70)
print(" SHAP 분석 요약")
print("="*70)

print("\n[1. 주요 피처 (Top 10)]")
for i, row in importance_df.head(10).iterrows():
    print(f"   {row['feature']:15} : {row['mean_abs_shap']:.4f}")

print("\n[2. 피처 그룹별 중요도]")
for group, imp in sorted(group_importance.items(), key=lambda x: -x[1]):
    print(f"   {group:20} : {imp:.4f}")

print("\n[3. 생성된 이미지]")
print(f"   - shap_summary_beeswarm.png    : 전체 피처 SHAP 분포")
print(f"   - shap_bar_importance.png      : 피처 중요도 막대그래프")
print(f"   - shap_waterfall_default.png   : 부도기업 개별 설명")
print(f"   - shap_waterfall_normal.png    : 정상기업 개별 설명")
print(f"   - shap_dependence_top6.png     : 상위 6개 피처 의존성")
print(f"   - shap_distribution_by_class.png : 부도/정상별 SHAP 분포")
print(f"   - shap_force_plot.png          : Force Plot")
print(f"   - shap_group_importance.png    : 피처 그룹별 중요도")

print("\n" + "="*70)

In [ ]:
# SHAP 값 저장 (추후 분석용)
np.save(OUTPUT_DIR / 'shap_values.npy', shap_values)
X_sample.to_csv(OUTPUT_DIR / 'shap_sample_data.csv', index=False)

print(f"SHAP 값 저장: {OUTPUT_DIR / 'shap_values.npy'}")
print(f"샘플 데이터 저장: {OUTPUT_DIR / 'shap_sample_data.csv'}")